## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

In [1]:
import pandas as pd
import numpy as np

from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

from scipy.stats import chi2_contingency

In [2]:
df = pd.read_csv('cookie_cats.csv')

In [3]:
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  object
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 2.2+ MB


1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [5]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# базові значения
p1 = 0.19
p2 = 0.20

# effect size
effect_size = proportion_effectsize(p1, p2)

# обʼєкт
analysis = NormalIndPower()

# розмір вибірки на одну групу
sample_size = analysis.solve_power(
    effect_size=effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
)

print("Розмір вибірки на одну групу:", round(sample_size))
print("Загальний розмір вибірки:", round(sample_size * 2))

Розмір вибірки на одну групу: 24638
Загальний розмір вибірки: 49276


Висновок: Для виявлення збільшення retention з 19% до 20% при рівні значущості 0.05 та потужності 80% необхідно приблизно 24 638 користувачів у кожній групі (або 49 276 загалом).
Це означає, що для виявлення такого невеликого ефекту потрібна досить велика вибірка

2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [6]:
df.groupby('version')['retention_7'].mean()

,retention_7
version,
gate_30,0.190201
gate_40,0.182000


Гіпотеза: Можна припустити, що версія гри з воротами на 30 рівні (gate_30) має краще утримання користувачів через 7 днів, ніж версія з воротами на 40 рівні (gate_40), оскільки середнє значення retention_7 є вищим.

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


Z-test:

In [7]:
# успіхи
success = [
    df[df['version'] == 'gate_30']['retention_7'].sum(),
    df[df['version'] == 'gate_40']['retention_7'].sum()
]

# розмір вибірок
nobs = [
    len(df[df['version'] == 'gate_30']),
    len(df[df['version'] == 'gate_40'])
]

# z-test
z_stat, p_value = proportions_ztest(success, nobs)

print("z statistic:", z_stat)
print("p-value:", p_value)

z statistic: 3.164358912748191
p-value: 0.001554249975614329


Довірчі інтервали:

In [8]:
# для gate_30
ci_30 = proportion_confint(
    success[0],
    nobs[0],
    alpha=0.05,
    method='normal'
)

# для gate_40
ci_40 = proportion_confint(
    success[1],
    nobs[1],
    alpha=0.05,
    method='normal'
)

print("95% CI для gate_30:", ci_30)
print("95% CI для gate_40:", ci_40)

95% CI для gate_30: (0.18656311652199903, 0.19383956804175934)
95% CI для gate_40: (0.17845430073314686, 0.18554578720019968)


Висоновок:
1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри? - Значення p-value = 0.00155. Оскільки p-value < 0.05, ми відхиляємо нульову гіпотезу. Це означає, що існує статистично значуща різниця між версіями гри.
2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже? - Довірчі інтервали для двох груп не перетинаються. Це свідчить про те, що різниця між групами є статистично значущою, і версія gate_30 має краще утримання користувачів.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


Гіпотеза:
H0: Версія гри та утримання на 7 день є незалежними.
H1: Версія гри та утримання на 7 день є залежними.

In [9]:
contingency_table = pd.crosstab(df['version'], df['retention_7'])
print(contingency_table)

retention_7  False  True 
version                  
gate_30      36198   8502
gate_40      37210   8279


In [10]:
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("chi2:", chi2)
print("p-value:", p_value)

chi2: 9.959086799559167
p-value: 0.0016005742679058301


Висновок:
Значення p-value = 0.0016. Оскільки p-value < 0.05, ми відхиляємо нульову гіпотезу. Це означає, що між версією гри та утриманням користувачів на 7 день існує статистично значуща залежність.